<a href="https://colab.research.google.com/github/erizz2/PHYS220-Project/blob/main/PHYS220proj_5-1-26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# import useful libraries
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import math
from scipy.integrate import quad
from IPython.display import HTML

# functions

def explicit_method(k, h, BC1, BC2, N):

    '''
    This function uses the explicit method of calculating the solution the the heat equation.

    Parameters:
    k (float): The time step.
    h (float): The spatial step.
    T (float): The temperature.
    N (int): The number of points.
    BC1 (float): The left boundary condition.
    BC2 (float): The right boundary condition.
    target (float): The accuracy condition.
    '''

    M = int(1000 / k)
    u = np.zeros([N+1, M+1], float)
    x = np.linspace(0, 100, N+1)

    gaussian = lambda x: 30000 * np.exp(-0.015*(x-50)**2)

    u[:,0] = gaussian(x) # initial spatial profile


    # boundary conditions
    u[0, :] = BC1
    u[N, :] = BC2

    #uprime = u.copy()
    r = k / h**2


    for n in range(M):
        for j in range(1, N):
            u[j, n+1] = (1 - 2*r)*u[j, n] + r*u[j-1, n] + r*u[j+1, n]

    return x, u

def implicit_method(k, h, BC1, BC2, N, target):

    '''
    This function uses the implicit method of calculating the solution the the heat equation.

    Parameters:
    k (float): The time step.
    h (float): The spatial step.
    T (float): The temperature.
    N (int): The number of points.
    BC1 (float): The left boundary condition.
    BC2 (float): The right boundary condition.
    target (float): The accuracy condition.
    '''

    M = int(1000 / k)
    u = np.zeros([N+1, M+1], float)
    x = np.linspace(0, 100, N+1)

    gaussian = lambda x: 3000 * np.exp(-0.01*(x-50)**2)

    u[:,0] = gaussian(x) # initial spatial profile


    # boundary conditions
    u[0, :] = BC1
    u[N, :] = BC2

    #uprime = u.copy()
    r = k / h**2

    for n in range(M):
        u[:, n+1] = u[:, n]
        u[0, n+1] = BC1
        u[N, n+1] = BC2

        delta = 1.0

        while delta > target:

          for j in range(1, N):
            uprime = u[:, n+1].copy()

            u[j, n+1] = 1 / (1 + 2*r) * (u[j, n] + r*u[j-1, n+1] + r*u[j+1, n+1])
          delta = np.max(np.abs(uprime - u[:, n+1]))
          #uprime = u.copy()


    return x, u
'''
def update(frame):
    line.set_data(x, phi[:, frame])
    return line,

def update_color_bar(frame):
        step = frame
        if step < phi.shape[1]:
            new_data = phi[:, step].reshape(1, -1)
            im.set_array(new_data)

            time_text.set_text(f"Timestep: {step}")

        return [im, time_text]
'''
def analytical_solution(L, alpha):
    '''
    This function computes the heat equation analytical solution.

    Parameters:
    L (float): The length of the rod.
    alpha (float): The thermal diffusivity of the material.
    '''

    t0=10 # maximum time
    gaussian = lambda x: 3000 * np.exp(-0.01*(x-0.5)**2)
    epsi_pepsi=10**(-6) # epsilon

    x_vals=np.linspace(0,L,100)
    t_vals=np.linspace(0,t0,100)

    u_xt_array = np.zeros((len(t_vals), len(x_vals)))
    for i, t in enumerate(t_vals):
        for j, x in enumerate(x_vals):
            u_xt_array[i, j] = u_xt(x, t, alpha)

    # Plot a few time slices
    time_indices = [0, 10, 25, 50, 75, 99]

    plt.figure()
    for t in time_indices:
        plt.plot(x_vals, u_xt_array[t, :], label=f"t={t_vals[t]:.2f}")

    plt.xlabel("x")
    plt.ylabel("u(x,t)")
    plt.title("Heat Equation Evolution (Snapshots)")
    plt.legend()
    plt.show()
    return

def simp(func,a,b,n):
    N = 1000
    h=(b-a)/N
    evens=0
    odds=0
    for k in range(1,N):
      if k%2==0:
        evens+=func(n,a+(k*h),b)
      else:
        odds+=func(n,a+(k*h),b)
    return (1/3)*h*(func(n,a,b)+func(n,b,b)+(4*odds)+(2*evens))

def B_n(n,L):
    return (2/L)*simp(func,0,L,n)

def func(n,x,L):
    return (gaussian(x)*np.sin((n*np.pi*x)/L))

def u_xt(x,t,L, alpha):
  total=0
  n = 1
  while True:
    B = B_n(n,L)
    current_term = B * np.sin((n*np.pi*x)/L) * np.exp(-(alpha*((n*np.pi)/L)**2)*t)

    if abs(current_term) < epsi_pepsi:
      break

    total+=current_term
    n += 1

  return total

def plot_xt(phi):
    plt.figure()
    plt.imshow(phi, origin='lower', cmap='inferno')
    plt.colorbar(label='Temperature (K)')
    plt.ylabel('x (m)')
    plt.xlabel('t (s)')
    plt.title("Heat vs. Time")
    plt.show()
    return

def plot_time_evolution(phi):
    fig, ax = plt.subplots()
    line, = ax.plot([], [])
    ax.set_xlabel("Position (m)")
    ax.set_ylabel("Temperature (K)")
    ax.set_xlim(0, 100)
    ax.set_ylim(np.min(phi), np.max(phi))

    def update(frame):
        line.set_data(x, phi[:, frame])
        return line,


    ani = FuncAnimation(fig, update, frames = range(0, phi.shape[1], 10), interval = 50, blit=False)


    plt.close(fig)
    display(HTML(ani.to_jshtml()))
    return

def plot_colorbar_time_evolution(phi):
    fig, ax = plt.subplots(figsize=(10, 4))
    rod_data = phi[:,0].reshape(1, -1)

    im = ax.imshow(rod_data, aspect=5.0, cmap='inferno')

    time_text = ax.text(0.5, 0.01, '', fontsize=12, color='white')
    ax.set_yticks([])
    ax.set_xlabel("Position (m)")
    ax.set_title("1D Heat Equation Evolution")
    plt.colorbar(im, orientation='horizontal', label='Temperature (K)', pad=0.3)

    def update_color_bar(frame):
        step = frame
        if step < phi.shape[1]:
            new_data = phi[:, step].reshape(1, -1)
            im.set_array(new_data)

            time_text.set_text(f"Timestep: {step}")

        return [im, time_text]

    ani = FuncAnimation(fig, update_color_bar, frames = range(0, phi.shape[1], 10), interval=35, blit=True, repeat=True)

    plt.close(fig)
    display(HTML(ani.to_jshtml()))


# main
if __name__ == "__main__":
    # analytical solution
    #analytical_solution(1.0, 0.01)

    # implicit method

    x, phi = explicit_method(1.0, 3.0, 0.0, 0.0, 100)

    # plotting
    plot_xt(phi)
    plot_time_evolution(phi)
    plot_colorbar_time_evolution(phi)
